# Multilingual-E5-Large: Text Embedding Research

In [ ]:
# !pip install sentence-transformers==3.0.1 torch datasets


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAME = "intfloat/multilingual-e5-large"
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded. Embedding dim: {model.get_sentence_embedding_dimension()}")


## 1. Sample pin corpus (RU/EN mix)

In [ ]:
pin_corpus = [
    # Russian pins
    "Закат на Байкале — невероятные оттенки оранжевого",
    "Уличная еда в Стамбуле: симит, балык экмек, турецкий чай",
    "Горы Алтая. Катунь в сентябре — бирюзовая вода, тишина",
    "Арт-объект в Москве, Артплей. Неоновые надписи ночью",
    "Цветущая сакура в Японии. Парк Синдзюку-гёэн, апрель",
    "Кофе в Тбилиси: маленькие кафе на серной бане",
    # English pins
    "Santorini sunset with caldera view — white-washed cliffs",
    "Street tacos in Mexico City: al pastor with pineapple",
    "Hiking the Dolomites: Tre Cime loop, August morning",
    "Tokyo street photography at night — neon and rain",
    "Specialty coffee in Melbourne laneway cafes",
    "Northern Lights over Tromsø, Norway",
]

queries = [
    "закат над озером",
    "уличная еда азия",
    "горный поход",
    "ночная фотография город",
    "кофе кафе путешествие",
]
print(f"Corpus: {len(pin_corpus)} pins, {len(queries)} queries")


## 2. Encode with query/passage prefix

In [ ]:
passage_texts = [f"passage: {p}" for p in pin_corpus]
query_texts   = [f"query: {q}" for q in queries]

t0 = time.perf_counter()
passage_embs = model.encode(passage_texts, normalize_embeddings=True, show_progress_bar=True)
query_embs   = model.encode(query_texts,   normalize_embeddings=True, show_progress_bar=True)
elapsed = time.perf_counter() - t0

print(f"Encoded {len(pin_corpus) + len(queries)} texts in {elapsed:.2f}s")
print(f"Passage embeddings shape: {passage_embs.shape}")
print(f"Query embeddings shape:   {query_embs.shape}")


## 3. Retrieval: top-3 pins per query

In [ ]:
sim_matrix = cosine_similarity(query_embs, passage_embs)

print("=" * 60)
for i, query in enumerate(queries):
    top3 = np.argsort(sim_matrix[i])[::-1][:3]
    print(f"
Query: {query!r}")
    for rank, idx in enumerate(top3, 1):
        print(f"  {rank}. [{sim_matrix[i, idx]:.3f}] {pin_corpus[idx]!r}")
print("=" * 60)


## 4. Similarity heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(sim_matrix, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(len(queries)))
ax.set_yticklabels(queries, fontsize=9)
ax.set_xticks(range(len(pin_corpus)))
ax.set_xticklabels([p[:35] + "..." for p in pin_corpus], rotation=45, ha="right", fontsize=8)
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("Query-Passage Cosine Similarity (multilingual-e5-large)")
plt.tight_layout()
plt.savefig("e5_heatmap.png", dpi=120)
plt.show()


## 5. Throughput benchmark (batch sizes)

In [ ]:
import random, string

def random_text(n=60):
    return " ".join("".join(random.choices(string.ascii_lowercase, k=7)) for _ in range(n // 7))

results = []
for batch_size in [1, 4, 8, 16, 32, 64]:
    texts = [f"passage: {random_text()}" for _ in range(batch_size)]
    t0 = time.perf_counter()
    for _ in range(5):
        model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
    elapsed = (time.perf_counter() - t0) / 5
    qps = batch_size / elapsed
    results.append({"batch_size": batch_size, "latency_s": elapsed, "qps": qps})
    print(f"batch={batch_size:3d}  latency={elapsed*1000:.1f}ms  qps={qps:.1f}")

df = pd.DataFrame(results)
df.plot(x="batch_size", y="qps", marker="o", title="E5-large throughput (CPU)")
plt.xlabel("Batch size")
plt.ylabel("Texts / second")
plt.grid(True)
plt.tight_layout()
plt.savefig("e5_throughput.png", dpi=120)
plt.show()


## Conclusions

| Metric | Value |
|--------|-------|
| Model | intfloat/multilingual-e5-large |
| Embedding dim | 1024 |
| Languages | 100+ (incl. RU, EN) |
| Prefix scheme | query: / passage: |

**Findings:**
- Cross-lingual retrieval works well: Russian queries correctly find English pins with semantically matching content
- Batch size 16-32 gives best CPU throughput
- For GPU inference use batch_size=64+

**Next steps:** compare with LaBSE and BGE-M3 in 
